# Link Prediction avec GNN — v3

**Fix principal — fuite de données** : dans la v2, le graphe `G` était construit sur *toutes* les arêtes positives de `train.txt`, y compris celles du split val. Les features structurelles d'une paire val `(u,v)` positive étaient donc calculées *avec* l'arête `(u,v)` déjà présente dans `G` → `common_neighbors` élevé, Jaccard élevé, etc. Le modèle voyait la réponse pendant la validation, d'où un AUC val de 0.977 artificiel.

**Correction** : split en premier, graphe construit uniquement sur les arêtes positives du train.

**Autres changements** :
- `SAGEConv` à la place de `GCNConv` (meilleure généralisation, pas de dépendance au degré)
- Features structurelles injectées **aussi** dans les features de nœuds avant l'encodeur
- Soumission en probas (déjà fait en v2)

In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch_geometric.data import Data
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

d:\Bazar\Travail Yann\CS\3A\MLNS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Chargement et split AVANT toute construction de graphe

In [2]:
train_full = pd.read_csv("data/train.txt", sep=" ", header=None)
train_full.columns = ["u", "v", "label"]

test = pd.read_csv("data/test.txt", sep=" ", header=None)
test.columns = ["u", "v"]

node_info = pd.read_csv("data/node_information.csv", header=None)
node_info = node_info.rename(columns={0: "node"})
node_features_raw = {
    int(row["node"]): row.drop("node").values.astype(np.float32)
    for _, row in node_info.iterrows()
}
feature_dim = len(next(iter(node_features_raw.values())))

# ─── SPLIT EN PREMIER ────────────────────────────────────────────────────────
# C'est le changement fondamental : on sépare train/val avant de construire
# quoi que ce soit, pour éviter que les features du val soient calculées
# avec les arêtes val dans le graphe.
train_df, val_df = train_test_split(
    train_full, test_size=0.2, stratify=train_full["label"], random_state=42
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f"Train : {len(train_df)} paires | Val : {len(val_df)} paires")
print(f"Ratio positifs train : {train_df['label'].mean():.2%}")

Train : 8396 paires | Val : 2100 paires
Ratio positifs train : 50.00%


## 2. Graphe de message-passing — uniquement sur les arêtes positives du TRAIN

On n'utilise **que** les arêtes positives de `train_df` (pas de `val_df`).  
Les features structurelles val/test seront donc calculées sans connaître les vrais liens du val.

In [3]:
# Graphe construit SEULEMENT sur les positifs du train split
G = nx.Graph()
train_pos = train_df[train_df["label"] == 1][["u", "v"]].values
G.add_edges_from(train_pos)

# On ajoute aussi les nœuds isolés présents dans train_df (négatifs)
for u, v in train_df[["u", "v"]].values:
    if not G.has_node(u): G.add_node(u)
    if not G.has_node(v): G.add_node(v)

# Idem pour val et test (les nœuds doivent exister pour avoir des features)
for u, v in val_df[["u", "v"]].values:
    if not G.has_node(u): G.add_node(u)
    if not G.has_node(v): G.add_node(v)
for u, v in test[["u", "v"]].values:
    if not G.has_node(u): G.add_node(u)
    if not G.has_node(v): G.add_node(v)

print(f"Nœuds dans G : {G.number_of_nodes()} | Arêtes (train positifs) : {G.number_of_edges()}")

Nœuds dans G : 3597 | Arêtes (train positifs) : 4198


## 3. Features structurelles

Calculées sur `G` qui ne contient **pas** les arêtes val/test → pas de fuite.

In [4]:
degree     = dict(G.degree())
components = {n: c for c, comp in enumerate(nx.connected_components(G)) for n in comp}
feat_norms = {
    node: np.linalg.norm(feat) + 1e-9
    for node, feat in node_features_raw.items()
}


def structural_features(u, v):
    deg_u = degree.get(u, 0)
    deg_v = degree.get(v, 0)

    if G.has_node(u) and G.has_node(v):
        nu    = set(G.neighbors(u))
        nv    = set(G.neighbors(v))
        inter = nu & nv
        union = nu | nv
        cn    = len(inter)
        jacc  = cn / len(union) if union else 0.0
        aa    = sum(
            1.0 / np.log(degree[w] + 1e-9)
            for w in inter if degree.get(w, 0) > 1
        )
        pa        = deg_u * deg_v
        same_comp = float(components.get(u, -1) == components.get(v, -2))
    else:
        cn, jacc, aa, pa, same_comp = 0, 0.0, 0.0, 0, 0.0

    if u in node_features_raw and v in node_features_raw:
        cosine = np.dot(node_features_raw[u], node_features_raw[v]) / (feat_norms[u] * feat_norms[v])
    else:
        cosine = 0.0

    return np.array([deg_u, deg_v, cn, jacc, aa, pa, same_comp, cosine], dtype=np.float32)


print("Calcul features structurelles...")
train_struct = np.stack([structural_features(r.u, r.v) for r in train_df.itertuples()])
val_struct   = np.stack([structural_features(r.u, r.v) for r in val_df.itertuples()])
test_struct  = np.stack([structural_features(r.u, r.v) for r in test.itertuples()])

scaler       = StandardScaler().fit(train_struct)
train_struct = scaler.transform(train_struct).astype(np.float32)
val_struct   = scaler.transform(val_struct).astype(np.float32)
test_struct  = scaler.transform(test_struct).astype(np.float32)

struct_dim = train_struct.shape[1]
print(f"Features structurelles : {struct_dim} dims")

Calcul features structurelles...
Features structurelles : 8 dims


## 4. Construction du graphe PyG

In [5]:
all_nodes   = sorted(G.nodes())
node_to_idx = {n: i for i, n in enumerate(all_nodes)}
num_nodes   = len(all_nodes)

# Features de nœuds = features brutes + features structurelles du nœud
# (degré normalisé, composante) — signal local utile pour SAGEConv
max_deg = max(degree.values()) + 1e-9
node_struct = np.array([
    [
        degree.get(n, 0) / max_deg,           # degré normalisé
        float(components.get(n, 0)),           # id composante
    ]
    for n in all_nodes
], dtype=np.float32)

x_raw = np.zeros((num_nodes, feature_dim), dtype=np.float32)
for node, idx in node_to_idx.items():
    if node in node_features_raw:
        x_raw[idx] = node_features_raw[node]

# Concaténation : features brutes + features structurelles locales
x_combined = np.concatenate([x_raw, node_struct], axis=1)
x_tensor   = torch.tensor(x_combined, dtype=torch.float)
full_feature_dim = x_tensor.shape[1]

edge_list = [
    (node_to_idx[u], node_to_idx[v])
    for u, v in G.edges()
    if u in node_to_idx and v in node_to_idx
]
edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)

data = Data(x=x_tensor, edge_index=edge_index, num_nodes=num_nodes)
print(data)
print(f"Dimension d'entrée du GNN : {full_feature_dim} (brutes {feature_dim} + struct locales {node_struct.shape[1]})")

Data(x=[3597, 934], edge_index=[2, 8396], num_nodes=3597)
Dimension d'entrée du GNN : 934 (brutes 932 + struct locales 2)


## 5. Tenseurs de paires

In [6]:
def pairs_to_tensors(df, struct_arr, node_to_idx, has_label=True):
    mask    = df["u"].isin(node_to_idx) & df["v"].isin(node_to_idx)
    idx     = np.where(mask.values)[0]
    valid   = df.iloc[idx]
    u_t     = torch.tensor([node_to_idx[u] for u in valid["u"]], dtype=torch.long)
    v_t     = torch.tensor([node_to_idx[v] for v in valid["v"]], dtype=torch.long)
    s_t     = torch.tensor(struct_arr[idx], dtype=torch.float)
    y_t     = torch.tensor(valid["label"].values, dtype=torch.float) if has_label else None
    return u_t, v_t, s_t, y_t

train_u, train_v, train_s, train_y = pairs_to_tensors(train_df, train_struct, node_to_idx)
val_u,   val_v,   val_s,   val_y   = pairs_to_tensors(val_df,   val_struct,   node_to_idx)
print(f"Paires train : {len(train_y)} | val : {len(val_y)}")

Paires train : 8396 | val : 2100


## 6. Modèles — SAGEConv + LinkPredictor

`SAGEConv` agrège en moyennant les voisins sans normaliser par le degré,
ce qui le rend plus robuste sur des graphes hétérogènes (hubs vs nœuds isolés).

In [7]:
class GNN_model(torch.nn.Module):
    def __init__(self, num_layers, input_size, hidden_size, output_size, dropout=0.4):
        super().__init__()
        self.convs = torch.nn.ModuleList()
        self.bns   = torch.nn.ModuleList()
        self.convs.append(SAGEConv(input_size, hidden_size))
        self.bns.append(torch.nn.BatchNorm1d(hidden_size))
        for _ in range(num_layers - 2):
            self.convs.append(SAGEConv(hidden_size, hidden_size))
            self.bns.append(torch.nn.BatchNorm1d(hidden_size))
        self.convs.append(SAGEConv(hidden_size, output_size))
        self.dropout = dropout

    def forward(self, x, edge_index):
        for conv, bn in zip(self.convs[:-1], self.bns):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return self.convs[-1](x, edge_index)


class LinkPredictor(torch.nn.Module):
    def __init__(self, emb_dim, struct_dim, hidden_dim=128, dropout=0.3):
        super().__init__()
        in_dim = emb_dim * 2 + struct_dim
        self.net = torch.nn.Sequential(
            torch.nn.Linear(in_dim, hidden_dim),
            torch.nn.BatchNorm1d(hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(hidden_dim, hidden_dim // 2),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, z_u, z_v, struct_feats):
        return self.net(torch.cat([z_u, z_v, struct_feats], dim=-1)).squeeze(-1)

## 7. Entraînement

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

NUM_LAYERS   = 3
HIDDEN_SIZE  = 128
EMB_SIZE     = 64
DROPOUT_GNN  = 0.4
DROPOUT_MLP  = 0.3
LR           = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS       = 600
PATIENCE     = 60

data    = data.to(device)
train_u, train_v, train_s, train_y = (
    train_u.to(device), train_v.to(device), train_s.to(device), train_y.to(device)
)
val_u, val_v, val_s, val_y = (
    val_u.to(device), val_v.to(device), val_s.to(device), val_y.to(device)
)

gnn       = GNN_model(NUM_LAYERS, full_feature_dim, HIDDEN_SIZE, EMB_SIZE, DROPOUT_GNN).to(device)
predictor = LinkPredictor(EMB_SIZE, struct_dim, HIDDEN_SIZE, DROPOUT_MLP).to(device)
optimizer = torch.optim.Adam(
    list(gnn.parameters()) + list(predictor.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=25, verbose=True
)
pos_weight = torch.tensor([(train_y == 0).sum() / (train_y == 1).sum()]).to(device)


def evaluate(z, u_idx, v_idx, s, y):
    with torch.no_grad():
        logits = predictor(z[u_idx], z[v_idx], s)
        loss   = F.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weight)
        probs  = torch.sigmoid(logits).cpu().numpy()
        auc    = roc_auc_score(y.cpu().numpy(), probs)
    return loss.item(), auc


best_val_auc  = 0.0
best_state    = None
epochs_no_imp = 0

for epoch in range(1, EPOCHS + 1):
    gnn.train(); predictor.train()
    optimizer.zero_grad()
    z      = gnn(data.x, data.edge_index)
    logits = predictor(z[train_u], z[train_v], train_s)
    loss   = F.binary_cross_entropy_with_logits(logits, train_y, pos_weight=pos_weight)
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        gnn.eval(); predictor.eval()
        z_eval          = gnn(data.x, data.edge_index)
        tr_loss, tr_auc = evaluate(z_eval, train_u, train_v, train_s, train_y)
        va_loss, va_auc = evaluate(z_eval, val_u,   val_v,   val_s,   val_y)
        scheduler.step(va_auc)
        print(
            f"Epoch {epoch:>3} | "
            f"Train loss {tr_loss:.4f} AUC {tr_auc:.4f} | "
            f"Val   loss {va_loss:.4f} AUC {va_auc:.4f}"
        )
        if va_auc > best_val_auc:
            best_val_auc  = va_auc
            epochs_no_imp = 0
            best_state    = {
                "gnn":       {k: v.cpu() for k, v in gnn.state_dict().items()},
                "predictor": {k: v.cpu() for k, v in predictor.state_dict().items()},
            }
        else:
            epochs_no_imp += 20
            if epochs_no_imp >= PATIENCE:
                print(f"Early stopping à l'epoch {epoch}.")
                break

print(f"\nMeilleur AUC validation (sans fuite) : {best_val_auc:.4f}")
print("Ce score devrait maintenant être proche de ce qu'on obtient sur Kaggle.")

Device : cpu


d:\Bazar\Travail Yann\CS\3A\MLNS\.venv\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch  20 | Train loss 0.5292 AUC 0.8795 | Val   loss 1.0180 AUC 0.5126
Epoch  40 | Train loss 0.2321 AUC 0.9696 | Val   loss 1.5503 AUC 0.5679
Epoch  60 | Train loss 0.1103 AUC 0.9949 | Val   loss 2.2586 AUC 0.5813
Epoch  80 | Train loss 0.0324 AUC 0.9994 | Val   loss 3.0561 AUC 0.5840
Epoch 100 | Train loss 0.0085 AUC 0.9999 | Val   loss 3.7585 AUC 0.5889
Epoch 120 | Train loss 0.0033 AUC 1.0000 | Val   loss 4.1783 AUC 0.5817
Epoch 140 | Train loss 0.0016 AUC 1.0000 | Val   loss 4.7217 AUC 0.5783
Epoch 160 | Train loss 0.0010 AUC 1.0000 | Val   loss 5.0455 AUC 0.5827
Early stopping à l'epoch 160.

Meilleur AUC validation (sans fuite) : 0.5889
Ce score devrait maintenant être proche de ce qu'on obtient sur Kaggle.


## 8. Inférence et soumission

Soumission en **probas continues** — indispensable si Kaggle évalue en AUC.

In [9]:
gnn.load_state_dict({k: v.to(device) for k, v in best_state["gnn"].items()})
predictor.load_state_dict({k: v.to(device) for k, v in best_state["predictor"].items()})
gnn.eval(); predictor.eval()

with torch.no_grad():
    z = gnn(data.x, data.edge_index)

test_s_tensor = torch.tensor(test_struct, dtype=torch.float, device=device)
u_list = [node_to_idx.get(int(r.u), -1) for r in test.itertuples()]
v_list = [node_to_idx.get(int(r.v), -1) for r in test.itertuples()]

scores = []
BATCH  = 512
with torch.no_grad():
    for start in range(0, len(test), BATCH):
        end   = min(start + BATCH, len(test))
        s_b   = test_s_tensor[start:end]
        batch = []
        for i, (ui, vi) in enumerate(zip(u_list[start:end], v_list[start:end])):
            if ui == -1 or vi == -1:
                batch.append(0.1)
            else:
                logit = predictor(z[ui].unsqueeze(0), z[vi].unsqueeze(0), s_b[i].unsqueeze(0))
                batch.append(torch.sigmoid(logit).item())
        scores.extend(batch)

test["score"] = scores
submit = pd.DataFrame({"ID": range(len(scores)), "Predicted": scores})
submit.to_csv("GNN_SAGE_structfeats.csv", index=False)

print(test[["u", "v", "score"]].head(10).to_string(index=False))
print(f"\nScore min : {min(scores):.4f}  max : {max(scores):.4f}  mean : {np.mean(scores):.4f}")

   u    v        score
3425 4524 1.419089e-05
1620 2617 6.066941e-04
4832 6317 9.981343e-01
4984 7298 1.236358e-04
 385 5481 2.280555e-03
1722 2930 1.372742e-05
1534 3330 9.561800e-01
5015 6354 2.071143e-05
 856 2504 9.796092e-09
 851 5515 3.343041e-04

Score min : 0.0000  max : 1.0000  mean : 0.2928
